<a href="https://colab.research.google.com/github/patilprajwal83094-arch/1MS25IS077--ASSIGNMENT-2/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [41]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [42]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [43]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section."
        "Documents can be related to any topic - sports, manga, food, music etc."
        "Have a minimum of 4 sentences with each document."
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
    "fill_in1":(),
    "fill_in2":(),
    "fill_in3":(),
    "fill_in4":()
}

In [44]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        # Ensure the content is a string and not an empty placeholder tuple
        if not isinstance(content, str):
            continue

        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))

[Source: sample_protocol]
Students must fill in documents in this section.Documents can be related to any topic - sports, manga, food, music etc.Have a minimum of 4 sentences with each document.Duplicate this for a minimum of 4 documents to test your agent.


In [45]:
# @title Ask gemini to search for something
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # Use the supported Gemini model name
    model_name="gemini-2.5-flash",
    # Use the correct search tool defined in the notebook
    tools=[search_documents],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-2.5-flash


In [46]:
import time
import os
from google import genai
from google.genai import types

# Explicitly pass the API key loaded from environment variables
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find the most relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]
        }
    )
])

def run_agent(question: str):
    """Runs the agent loop for a user query using the search_documents tool."""
    print(f"User Question: {question}")

    chat = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=(
                "You are a chef training at a Michelin star restaurant. "
                "When you need specific information about temperatures, "
                "ingredients to use, cook time etc. use the search_documents tool. "
                "Only answer based on what you find in the documents. "
                "If the documents do not contain the answer, say so clearly."
            ),
            tools=[search_tool],
            temperature=0.0
        )
    )

    response = chat.send_message(question)

    # Agent Loop: Handle function calls if the model requests them
    while response.function_calls:
        for call in response.function_calls:
            if call.name == "search_documents":
                print(f"[Agent action]: Calling search_documents tool with args: {call.args}")
                args = dict(call.args)
                # Call our python function with the provided arguments
                result = search_documents(**args)
                print(f"[Tool result]: Obtained {len(result)} characters.")

                # Send the tool output back to the model
                response = chat.send_message(
                    types.Part.from_function_response(
                        name=call.name,
                        response={"result": result}
                    )
                )
            else:
                print(f"Unknown function call requested: {call.name}")
                break

    print(f"\n[Final Answer]:\n{response.text}")

In [47]:
import time
from google.genai.errors import APIError

# Updated questions to satisfy cell test validation conditions (>= 4 questions, no default placeholder)
questions = [
    "What is the recommended cooking temperature for medium-rare steak?",
    "How long should a standard pasta dough rest before rolling?",
    "How many ghosts eat carrots everyday?",
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(2)  # Reduced delay to make execution run faster
    print("\n" + "=" * 60)
    try:
        run_agent(q)
    except APIError as e:
        print(f"\n[Error]: Google GenAI API error encountered for this query.\n{e}")
        print("Skipping to the next question...")
    except Exception as e:
        print(f"\n[Unexpected Error]: {e}")
        print("Skipping to the next question...")


User Question: What is the recommended cooking temperature for medium-rare steak?

[Error]: Google GenAI API error encountered for this query.
400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
Skipping to the next question...

User Question: How long should a standard pasta dough rest before rolling?

[Error]: Google GenAI API error encountered for this query.
400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'AP

In [48]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)